# Assignment 7: Noun Phrase Chunking Using Hidden Markov Models (HMM)

## 1. Problem Statement & Background

In Natural Language Processing (NLP), **Noun Phrase (NP) Chunking** (or shallow parsing) is the task of identifying non-overlapping segments of a sentence that correspond to noun phrases. 

Each word in a sentence is tagged using the standard **BIO (Beginning, Inside, Outside)** scheme:
- **`B-NP`**: The **Beginning** word of a noun phrase (e.g., *"The"* in *"The red car"*).
- **`I-NP`**: A word **Inside** a noun phrase that follows `B-NP` (e.g., *"red"* and *"car"* in *"The red car"*).
- **`O`**: Words **Outside** any noun phrase (e.g., verbs, punctuation, prepositions that are not part of an NP).

### Assignment Tasks:
1. **HMM Training with Add-K Smoothing**: Estimate transition and emission probabilities from the training set, applying Add-K smoothing for zero counts.
2. **Forward Algorithm**: Compute the total observation probability $P(\text{Sentence})$ for **each test sentence**.
3. **Viterbi Decoding**: Implement Viterbi decoding in log-space to determine the optimal sequence of NP tags for any given sentence.
4. **Evaluation**: Evaluate the predictions on the test dataset using the [`seqeval`](https://pypi.org/project/seqeval/) library and report entity-level Precision, Recall, and F1-score.

## 2. Environment Setup & Dependencies

We install and import `seqeval` along with standard numerical and data-handling libraries (`numpy`, `pandas`, `math`, `collections`).

In [8]:
# Install seqeval for sequence labeling evaluation
!pip install seqeval

import os
import math
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score

print("Libraries successfully imported!")

Libraries successfully imported!



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Data Loading & Exploratory Data Analysis (EDA)

The dataset is stored in standard CoNLL format where:
- Each non-empty line has a token and its NP tag separated by whitespace: `Word <TAB/SPACE> Tag`.
- Empty lines delimit sentence boundaries.

We implement `load_data(file_path)` to parse this format into lists of tokenized sentences and corresponding tag sequences.

In [9]:
def load_data(file_path):
    """
    Reads a CoNLL-style tagged corpus file and returns:
    - sentences: List of lists of words
    - tags: List of lists of corresponding tags
    """
    sentences = []
    tags = []
    curr_sent = []
    curr_tags = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if curr_sent:
                    sentences.append(curr_sent)
                    tags.append(curr_tags)
                    curr_sent = []
                    curr_tags = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    word = parts[0]
                    tag = parts[-1]
                    curr_sent.append(word)
                    curr_tags.append(tag)
                    
        if curr_sent:
            sentences.append(curr_sent)
            tags.append(curr_tags)
            
    return sentences, tags

# Locate train and test files (handling both relative and absolute paths)
train_path = os.path.join("Data", "train_NP.txt")
if not os.path.exists(train_path):
    train_path = r"C:\Users\krish\Desktop\LLMS\LAB7(10-9-26)\Data\train_NP.txt"

test_path = os.path.join("Data", "test_NP.txt")
if not os.path.exists(test_path):
    test_path = r"C:\Users\krish\Desktop\LLMS\LAB7(10-9-26)\Data\test_NP.txt"

train_sentences, train_tags = load_data(train_path)
test_sentences, test_tags = load_data(test_path)

print(f"[Dataset Statistics]")
print(f"  Training sentences: {len(train_sentences):,} | Training tokens: {sum(len(s) for s in train_sentences):,}")
print(f"  Test sentences:     {len(test_sentences):,} | Test tokens:     {sum(len(s) for s in test_sentences):,}")

[Dataset Statistics]
  Training sentences: 823 | Training tokens: 19,172
  Test sentences:     77 | Test tokens:     1,896


### Inspecting Sample Data and Tag Distribution
Let us examine the distribution of tags across the training and test sets and preview a sample sentence.

In [10]:
# Tag counts in training and test
train_tag_counts = Counter(tag for sent_tags in train_tags for tag in sent_tags)
test_tag_counts = Counter(tag for sent_tags in test_tags for tag in sent_tags)

df_tags = pd.DataFrame({
    'Train Count': train_tag_counts,
    'Test Count': test_tag_counts
}).fillna(0).astype(int)
df_tags['Train %'] = (df_tags['Train Count'] / df_tags['Train Count'].sum() * 100).round(2)
df_tags['Test %'] = (df_tags['Test Count'] / df_tags['Test Count'].sum() * 100).round(2)

print("Tag Frequencies:")
display(df_tags)

# Preview first sentence
sample_df = pd.DataFrame({
    'Token': train_sentences[0],
    'Tag': train_tags[0]
})
print("\nSample Sentence 1 (Training):")
display(sample_df.T)

Tag Frequencies:


,Train Count,Test Count,Train %,Test %
B-NP,5051,475,26.35,25.05
I-NP,5727,579,29.87,30.54
O,8394,842,43.78,44.41



Sample Sentence 1 (Training):


,0,1,2,3,4,5,6,7,8,9,...,18,19,20,21,22,23,24,25,26,27
Token,Rockwell,International,Corp.,'s,Tulsa,unit,said,it,signed,a,...,to,provide,structural,parts,for,Boeing,'s,747,jetliners,.
Tag,B-NP,I-NP,I-NP,B-NP,I-NP,I-NP,O,B-NP,O,B-NP,...,O,O,B-NP,I-NP,O,B-NP,B-NP,I-NP,I-NP,O


## 4. HMM Formulation & Training with Add-K Smoothing

### Hidden Markov Model Components:
- **Hidden States ($S$)**: The noun phrase tags: $S = \{\text{B-NP}, \text{I-NP}, \text{O}\}$.
  We also include special boundary states: `START` ($t=0$) and `END` ($t=T+1$).
- **Observations ($V$)**: All unique words in the training vocabulary.
- **Transition Probability $A_{ij} = P(s_t = j \mid s_{t-1} = i)$**: The probability of transitioning from tag $i$ to tag $j$.
- **Emission Probability $B_j(w) = P(w_t = w \mid s_t = j)$**: The probability of emitting word $w$ from tag $j$.

### Add-K Laplace Smoothing
In realistic testing, unseen transitions or out-of-vocabulary (OOV) words occur. Naive maximum likelihood estimates yield $0.0$, causing sequence probability underflow. Add-$k$ smoothing ($k > 0$) distributes probability mass to unseen events:

1. **Transition Probability**:
$$
P(s_j \mid s_i) = \frac{C(s_i, s_j) + k}{C(s_i) + k \cdot (|S_{\text{dest}}|)}
$$
where $|S_{\text{dest}}|$ is the number of valid target states ($|S|$ tags + `END` for intermediate tags; $|S|$ tags for `START`).

2. **Emission Probability**:
$$
P(w \mid s_j) = \frac{C(s_j, w) + k}{C(s_j) + k \cdot (|V| + 1)}
$$
where $|V|$ is the training vocabulary size, and $+1$ accounts for an unknown token placeholder (`<UNK>`).

In [11]:
# Set smoothing constant (k)
k = 0.1

# Counting data structures
tag_counts = defaultdict(int)
tag_bigram_counts = defaultdict(lambda: defaultdict(int))
tag_word_counts = defaultdict(lambda: defaultdict(int))
vocab = set()
states_set = set()

# Accumulate counts from training sentences
for sent, tags in zip(train_sentences, train_tags):
    prev_tag = 'START'
    tag_counts['START'] += 1
    
    for word, tag in zip(sent, tags):
        vocab.add(word)
        states_set.add(tag)
        
        tag_counts[tag] += 1
        tag_bigram_counts[prev_tag][tag] += 1
        tag_word_counts[tag][word] += 1
        
        prev_tag = tag
        
    tag_bigram_counts[prev_tag]['END'] += 1

states = sorted(list(states_set))
vocab_size = len(vocab)

print(f"Training summary:")
print(f"  Hidden states (N): {states}")
print(f"  Vocabulary size (|V|): {vocab_size:,}")
print(f"  Smoothing constant (k): {k}")

def get_transition_prob(prev_tag, curr_tag, k=k):
    """Calculates P(curr_tag | prev_tag) using Add-k smoothing."""
    count_pair = tag_bigram_counts[prev_tag].get(curr_tag, 0)
    count_prev = tag_counts[prev_tag]
    
    # START can only transition into standard states
    # Standard states can transition into standard states + END
    num_destinations = len(states) if prev_tag == 'START' else len(states) + 1
    
    return (count_pair + k) / (count_prev + k * num_destinations)

def get_emission_prob(tag, word, k=k):
    """Calculates P(word | tag) using Add-k smoothing (handles unseen words)."""
    count_tw = tag_word_counts[tag].get(word, 0)
    count_tag = tag_counts[tag]
    return (count_tw + k) / (count_tag + k * (vocab_size + 1))

Training summary:
  Hidden states (N): ['B-NP', 'I-NP', 'O']
  Vocabulary size (|V|): 4,570
  Smoothing constant (k): 0.1


### Transition Probability Matrix Visualization
Let us inspect the learned transition probabilities. Notice that:
- $P(\text{I-NP} \mid \text{START})$ is very low because a noun phrase cannot start with an inside token.
- $P(\text{I-NP} \mid \text{B-NP})$ is high because multi-word noun phrases follow a beginning token with inside tokens.

In [12]:
# Build DataFrame of transition probabilities
trans_rows = ['START'] + states
trans_cols = states + ['END']

matrix_data = []
for r in trans_rows:
    row_vals = [get_transition_prob(r, c) for c in trans_cols]
    matrix_data.append(row_vals)

df_transitions = pd.DataFrame(matrix_data, index=trans_rows, columns=trans_cols)
print("Transition Probability Matrix P(Next_State | Prev_State):")
display(df_transitions.round(4))

Transition Probability Matrix P(Next_State | Prev_State):


,B-NP,I-NP,O,END
START,0.6572,0.0001,0.3426,0.0001
B-NP,0.0295,0.6955,0.2748,0.0002
I-NP,0.0458,0.3866,0.5652,0.0025
O,0.4883,0.0000,0.4154,0.0963


## 5. Task 1: The Forward Algorithm

The **Forward Algorithm** calculates the total probability of an observed sentence $W = (w_1, w_2, \dots, w_T)$ by summing over all possible hidden state sequences:
$$
P(W) = \sum_{\mathbf{s}} P(W, \mathbf{s})
$$

> **Numerical Note on Log-Space:** Because multiplying probabilities over long sequences ($T > 30$) can cause floating-point underflow to $0.0$, we compute both:
> 1. **Raw Probability $P(W)$**: Direct forward sum.
> 2. **Log Probability $\log P(W)$**: Computed using the log-sum-exp trick to maintain exact numerical precision.

In [13]:
def log_sum_exp(log_probs):
    """Numerically stable calculation of log(sum(exp(x)))."""
    max_val = max(log_probs)
    if max_val == -float('inf'):
        return -float('inf')
    return max_val + math.log(sum(math.exp(x - max_val) for x in log_probs))

def forward_algorithm(sentence):
    """
    Computes the total probability P(Sentence) and log P(Sentence)
    for a given sentence using the Forward algorithm.
    Returns: (raw_probability, log_probability)
    """
    if not sentence:
        return 0.0, -float('inf')
    
    # --- 1. Raw Probability Computation ---
    alpha = {}
    word0 = sentence[0]
    for state in states:
        alpha[state] = get_transition_prob('START', state) * get_emission_prob(state, word0)
        
    for t in range(1, len(sentence)):
        word = sentence[t]
        new_alpha = {}
        for curr_state in states:
            emit_p = get_emission_prob(curr_state, word)
            sum_prev = sum(alpha[prev_state] * get_transition_prob(prev_state, curr_state) for prev_state in states)
            new_alpha[curr_state] = emit_p * sum_prev
        alpha = new_alpha
        
    raw_prob = sum(alpha[s] * get_transition_prob(s, 'END') for s in states)
    
    # --- 2. Stable Log-Space Computation (Log-Sum-Exp) ---
    log_alpha = {}
    for state in states:
        log_alpha[state] = math.log(get_transition_prob('START', state)) + math.log(get_emission_prob(state, word0))
        
    for t in range(1, len(sentence)):
        word = sentence[t]
        new_log_alpha = {}
        for curr_state in states:
            log_emit = math.log(get_emission_prob(curr_state, word))
            prev_terms = [log_alpha[prev_state] + math.log(get_transition_prob(prev_state, curr_state)) for prev_state in states]
            new_log_alpha[curr_state] = log_emit + log_sum_exp(prev_terms)
        log_alpha = new_log_alpha
        
    end_terms = [log_alpha[s] + math.log(get_transition_prob(s, 'END')) for s in states]
    log_prob = log_sum_exp(end_terms)
    
    return raw_prob, log_prob

### Calculating Forward Probability for Each Test Sentence
As specified in Task 1, we execute the Forward algorithm on **every sentence** in the test dataset and summarize the output.

In [14]:
print(f"Running Forward Algorithm across all {len(test_sentences)} test sentences...")

forward_results = []
for idx, sent in enumerate(test_sentences):
    p_raw, p_log = forward_algorithm(sent)
    forward_results.append({
        'Sentence_ID': idx + 1,
        'Length': len(sent),
        'Preview': " ".join(sent[:8]) + ("..." if len(sent) > 8 else ""),
        'P(Sentence)': p_raw,
        'Log_P(Sentence)': p_log
    })

df_forward = pd.DataFrame(forward_results)

print("\nForward Algorithm Results (First 10 Test Sentences):")
display(df_forward.head(10))

print(f"\nSummary across all {len(test_sentences)} test sentences:")
print(f"  Mean Log Probability: {df_forward['Log_P(Sentence)'].mean():.4f}")
print(f"  Min Log Probability:  {df_forward['Log_P(Sentence)'].min():.4f}")
print(f"  Max Log Probability:  {df_forward['Log_P(Sentence)'].max():.4f}")

Running Forward Algorithm across all 77 test sentences...

Forward Algorithm Results (First 10 Test Sentences):


,Sentence_ID,Length,Preview,P(Sentence),Log_P(Sentence)
0,1,37,Confidence in the pound is widely expected to...,2.028768e-120,-275.602782
1,2,27,Chancellor of the Exchequer Nigel Lawson 's re...,3.309907e-88,-201.430568
2,3,29,But analysts reckon underlying support for ste...,1.932425e-98,-224.994564
3,4,36,This has increased the risk of the government...,9.542681e-108,-246.423416
4,5,31,`` The risks for sterling of a bad...,5.921827e-102,-233.085034
5,6,36,`` If there is another bad trade number...,2.098242e-115,-264.056186
6,7,39,"Forecasts for the trade figures range widely ,...",5.708943e-122,-279.173347
7,8,25,The August deficit and the # 2.2 billion...,6.939586e-83,-189.177321
8,9,28,"Sanjay Joshi , European economist at Baring Br...",5.232519e-94,-214.788106
9,10,24,"At the same time , he remains fairly...",5.944773e-79,-180.121710



Summary across all 77 test sentences:
  Mean Log Probability: -179.5349
  Min Log Probability:  -350.5360
  Max Log Probability:  -40.3104


## 6. Task 2: Viterbi Decoding

While the Forward algorithm sums probabilities across all possible paths, **Viterbi Decoding** finds the single **most probable sequence of hidden states** (tags) $\hat{\mathbf{s}} = (s_1, \dots, s_T)$ for a sentence:
$$
\hat{\mathbf{s}} = \arg\max_{\mathbf{s}} P(\mathbf{s}, W)
$$

To prevent numerical underflow, the Viterbi algorithm is formulated using **log-probabilities** ($V_t(j) = \max_{\mathbf{s}_{1:t-1}} \log P(s_1, \dots, s_t=j, w_1, \dots, w_t)$):

1. **Initialization ($t = 1$)**:
   $$
   V_1(j) = \log P(j \mid \text{START}) + \log P(w_1 \mid j) \quad \forall j \in S
   $$
   $$
   \text{Backpointer}_1(j) = \text{START}
   $$

2. **Recursion ($t = 2, \dots, T$)**:
   $$
   V_t(j) = \max_{i \in S} \Big( V_{t-1}(i) + \log P(j \mid i) \Big) + \log P(w_t \mid j) \quad \forall j \in S
   $$
   $$
   \text{Backpointer}_t(j) = \arg\max_{i \in S} \Big( V_{t-1}(i) + \log P(j \mid i) \Big)
   $$

3. **Termination ($t = T$)**:
   $$
   \hat{s}_T = \arg\max_{i \in S} \Big( V_T(i) + \log P(\text{END} \mid i) \Big)
   $$

4. **Backtracking**:
   $$
   \hat{s}_t = \text{Backpointer}_{t+1}(\hat{s}_{t+1}) \quad \text{for } t = T-1, T-2, \dots, 1
   $$

In [15]:
def viterbi_decoding(sentence):
    """
    Predicts the optimal sequence of NP tags for a given sentence
    using Viterbi decoding in log-space.
    """
    if not sentence:
        return []
    
    viterbi = []      # List of dicts storing max log-prob at each position
    backpointers = [] # List of dicts storing argmax previous state
    
    # --- 1. Initialization (t = 0) ---
    v0 = {}
    bp0 = {}
    word0 = sentence[0]
    for state in states:
        log_trans = math.log(get_transition_prob('START', state))
        log_emit = math.log(get_emission_prob(state, word0))
        v0[state] = log_trans + log_emit
        bp0[state] = 'START'
        
    viterbi.append(v0)
    backpointers.append(bp0)
    
    # --- 2. Recursion (t = 1 ... T-1) ---
    for t in range(1, len(sentence)):
        word = sentence[t]
        vt = {}
        bpt = {}
        for curr_state in states:
            log_emit = math.log(get_emission_prob(curr_state, word))
            
            # Maximize over previous state
            best_prob, best_prev = max(
                (viterbi[t-1][prev_state] + math.log(get_transition_prob(prev_state, curr_state)) + log_emit, prev_state)
                for prev_state in states
            )
            vt[curr_state] = best_prob
            bpt[curr_state] = best_prev
            
        viterbi.append(vt)
        backpointers.append(bpt)
        
    # --- 3. Termination ---
    best_final_prob, best_final_state = max(
        (viterbi[-1][state] + math.log(get_transition_prob(state, 'END')), state)
        for state in states
    )
    
    # --- 4. Backtracking ---
    best_path = [best_final_state]
    for t in range(len(sentence) - 1, 0, -1):
        prev_state = backpointers[t][best_path[0]]
        best_path.insert(0, prev_state)
        
    return best_path

### Predicting NP Tags on All Test Sentences
We apply the Viterbi algorithm to predict tags for every test sentence.

In [16]:
print(f"Predicting tags for {len(test_sentences)} test sentences using Viterbi decoding...")
predicted_tags = [viterbi_decoding(sent) for sent in test_sentences]
print("Decoding completed successfully!")

Predicting tags for 77 test sentences using Viterbi decoding...
Decoding completed successfully!


## 7. Task 3: Evaluation using `seqeval`

In chunking tasks, standard accuracy across individual tokens is misleading because it does not measure whether the complete **chunk boundaries** were identified correctly. 

The [`seqeval`](https://pypi.org/project/seqeval/) library evaluates chunk spans in the BIO format:
- A **True Positive (TP)** requires both correct entity type (`NP`) and exact boundary match (`B-NP` followed by exact `I-NP` span).
- Computes chunk-level **Precision**, **Recall**, and **F1-Score**.

In [17]:
# Compute overall seqeval metrics
prec = precision_score(test_tags, predicted_tags)
rec = recall_score(test_tags, predicted_tags)
f1 = f1_score(test_tags, predicted_tags)
acc = accuracy_score(test_tags, predicted_tags)

print("===================================================")
print("           HMM EVALUATION RESULTS (seqeval)        ")
print("===================================================")
print(f"  Chunk-Level Precision: {prec:.4f} ({prec * 100:.2f}%)")
print(f"  Chunk-Level Recall:    {rec:.4f} ({rec * 100:.2f}%)")
print(f"  Chunk-Level F1-Score:  {f1:.4f} ({f1 * 100:.2f}%)")
print(f"  Token-Level Accuracy:  {acc:.4f} ({acc * 100:.2f}%)")
print("===================================================\n")

print("Detailed Classification Report:")
print(classification_report(test_tags, predicted_tags, digits=4))

           HMM EVALUATION RESULTS (seqeval)        
  Chunk-Level Precision: 0.6679 (66.79%)
  Chunk-Level Recall:    0.7537 (75.37%)
  Chunk-Level F1-Score:  0.7082 (70.82%)
  Token-Level Accuracy:  0.8613 (86.13%)

Detailed Classification Report:
              precision    recall  f1-score   support

          NP     0.6679    0.7537    0.7082       475

   micro avg     0.6679    0.7537    0.7082       475
   macro avg     0.6679    0.7537    0.7082       475
weighted avg     0.6679    0.7537    0.7082       475



## 8. Qualitative Inspection / Error Analysis

Let us inspect the model's predictions on sample sentences side-by-side with ground-truth tags to observe where the HMM succeeds and where chunking boundary errors occur.

In [18]:
def display_sentence_prediction(sent_idx):
    words = test_sentences[sent_idx]
    true_t = test_tags[sent_idx]
    pred_t = predicted_tags[sent_idx]
    
    df_inspect = pd.DataFrame({
        'Token': words,
        'Ground Truth': true_t,
        'Viterbi Prediction': pred_t,
        'Match': ['✓' if t == p else '✗' for t, p in zip(true_t, pred_t)]
    })
    print(f"--- Test Sentence {sent_idx + 1} Inspection ---")
    display(df_inspect)

# Display predictions for first 2 test sentences
display_sentence_prediction(0)
display_sentence_prediction(1)

--- Test Sentence 1 Inspection ---


,Token,Ground Truth,Viterbi Prediction,Match
0,Confidence,B-NP,B-NP,✓
1,in,O,O,✓
2,the,B-NP,B-NP,✓
3,pound,I-NP,I-NP,✓
4,is,O,O,✓
5,widely,O,B-NP,✗
6,expected,O,O,✓
7,to,O,O,✓
8,take,O,O,✓
9,another,B-NP,B-NP,✓


--- Test Sentence 2 Inspection ---


,Token,Ground Truth,Viterbi Prediction,Match
0,Chancellor,O,B-NP,✗
1,of,O,O,✓
2,the,B-NP,B-NP,✓
3,Exchequer,I-NP,I-NP,✓
4,Nigel,O,I-NP,✗
5,Lawson,O,O,✓
6,'s,B-NP,B-NP,✓
7,restated,I-NP,I-NP,✓
8,commitment,I-NP,I-NP,✓
9,to,O,O,✓


## 9. Conclusion & Summary

1. **Add-K Smoothing ($k = 0.1$)**: Effectively handled out-of-vocabulary words and unseen transitions without zero-probability errors.
2. **Forward Algorithm**: Evaluated the probability of each sentence in the test dataset, computing exact raw probabilities for short sentences and stable log-probabilities for all sentences.
3. **Viterbi Decoding**: Correctly decoded optimal tag sequences by maximizing log joint probabilities using dynamic programming and backtracking.
4. **Evaluation with `seqeval`**: Evaluated span-level precision, recall, and F1-score on the NP identification task as requested in the assignment.